## CS587 – Software Project Management (Phase 2)
## Cloud-Based Intelligent Resource Allocation System for Campus Facilities

**Domain:** Education

**Framework:** LangChain

**Model:** Claude Sonnet 4.6 (claude-sonnet-4-6)

**Approach:** Round Robin Sequential Chain


In [1]:
import os
print("Current directory:", os.getcwd())
print(".env exists:", os.path.exists(".env"))
print("Files in directory:", os.listdir("."))

Current directory: c:\Users\nisha\Downloads\SPM-Phase-2\SPM-Phase-2
.env exists: True
Files in directory: ['.env', 'Exp1_AutoGen_GPT4.1_mini.ipynb', 'Exp2_LangChain_Sonnet46.ipynb', 'Exp3_Ollama_Llama3.ipynb', 'Experiment_Results_1', 'Experiment_Results_2', 'Experiment_Results_3', 'venv']


In [3]:
import os
key = os.environ.get("ANTHROPIC_API_KEY", "NOT FOUND")
print(f"Key found: {key[:15]}...{key[-4:] if len(key) > 15 else ''}")
print(f"Key length: {len(key)}")

Key found: sk-ant-api03-ZO...hwAA
Key length: 108


#### Install Dependencies

In [4]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "langchain", "langchain-anthropic", "anthropic", "tiktoken",
    "--quiet"
])
print("Libraries installed.")

Libraries installed.


#### Configuration & API Key

In [5]:
import os, json, time, re
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("anthropic").setLevel(logging.ERROR)
logging.getLogger("langchain").setLevel(logging.ERROR)

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
import tiktoken

with open(".env") as f:
    for line in f:
        line = line.strip()
        if line and "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found.")
print("API Key loaded.")

MODEL = "claude-sonnet-4-6"

llm = ChatAnthropic(
    model=MODEL,
    anthropic_api_key=ANTHROPIC_API_KEY,
    temperature=0.4,
    max_tokens=6000,
)

print("API Key loaded.")

API Key loaded.
API Key loaded.


#### Project Domain & Scrum Configuration

In [6]:
DOMAIN = """
PROJECT: Cloud-Based Intelligent Resource Allocation System for Campus Facilities
DOMAIN: Education
DESCRIPTION: Students and faculty book rooms, labs, and equipment online.
Administrators view real-time occupancy dashboards. AI/ML predicts demand and
auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook,
and SSO. Sends booking/conflict notifications. Generates utilization reports.
Supports 5,000+ concurrent users and maintains 99.9% uptime.
STAKEHOLDERS: Students, Faculty, Facility Admins, IT Department, University Management
FACILITIES: Classrooms, Computer Labs, Research Labs, Conference Rooms, Study Rooms
"""

SCRUM_CONFIG = """
SCRUM FRAMEWORK:
- Sprint Duration: 2 weeks (10 business days)
- Total Sprints: 4 sprints
- Sprint Velocity: 40 story points per sprint
- Story Point Scale: Fibonacci (1, 2, 3, 5, 8, 13, 21)
- Definition of Done: Code reviewed, unit tested, integrated, demo-ready
- Ceremonies: Sprint Planning, Daily Scrum, Sprint Review, Sprint Retrospective
- Team: Scrum Master, Product Owner, Developers, QA, DevOps, UI/UX Designer
"""
print("Domain and Scrum config loaded.")

Domain and Scrum config loaded.


#### Define Scrum AI Agents

In [7]:
AGENT_SYSTEM_PROMPTS = {

"Product_Owner": """IMPORTANT: You are ONLY the Product Owner.
IMPORTANT: Be concise and structured. Limit your response to essential content only. Do not repeat information. Use tables where possible. Aim for 2000-3000 words maximum.
ONLY provide YOUR role's specific output: Product Backlog, User Stories, Story Points, Sprint Assignments, and Definition of Done.
Start your response with: '## PRODUCT OWNER OUTPUT'

You are the Product Owner in a Scrum-based campus facility system project.
Responsibilities:
- Create a prioritized Product Backlog with at least 20 user stories
- Each story must include: ID, user story, acceptance criteria, priority, and story points
- Group stories into Epics (Authentication, Booking, AI/ML, Notifications, Reporting, Admin)
- Use ONLY Fibonacci story points: 3, 5, or 8. Never use 13 or 21 under any circumstances.
- If a story feels complex enough for 13 points, split it into two stories of 5+8 or cap it at 8.
- Every sprint MUST total between 36 and 40 SP — no exceptions.
- Plan 4 sprints using velocity = 40 story points per sprint
- Define Definition of Done (DoD)

Effort Calculation:
1. Calculate Total Work = sum of ALL story points one by one
   Add each story point value individually and double check your addition
2. Use Velocity = 40 story points/sprint
3. Compute Effort: Effort (sprints) = Total Work / Velocity

IMPORTANT:
- You MUST use story points ONLY
- You MUST show the actual numeric calculation step by step
- You MUST verify your total by adding numbers twice
- Output this section clearly as:
  - Total Work = ___ story points
  - Velocity = 40 story points/sprint
  - Effort = ___ sprints
- At the end of your response, include a separate section titled "Effort Estimate"
CRITICAL: You MUST complete your Effort Estimate section fully before ending your response. Never cut off mid-sentence or mid-table.

Output: structured markdown tables with calculations and detailed explanations.""",

"Scrum_Master": """IMPORTANT: You are ONLY the Scrum Master.
IMPORTANT: Be concise and structured. Limit your response to essential content only. Do not repeat information. Use tables where possible. Aim for 2000-3000 words maximum.
ONLY provide YOUR role's specific output: Sprint Ceremonies, Sprint Goals for all 4 sprints, Velocity Tracking, and Burndown Plan.
Start your response with: '## SCRUM MASTER OUTPUT'

You are a Scrum Master in a Scrum-based campus facility project.
Responsibilities:
- Define Scrum ceremonies: Sprint Planning, Daily Scrum, Sprint Review, Sprint Retrospective
- Each sprint MUST have EXACTLY 10 Daily Scrums — never 9, never 8.
- Per sprint: 1 Planning + 10 Daily Scrums + 1 Review + 1 Retrospective = 13 ceremonies
- 4 sprints × 13 = 52 total ceremonies (this number is fixed and non-negotiable)
- Effort = 52 / 3 = 17.33 days (always state this exact number)
- Create sprint goals for Sprint 1, Sprint 2, Sprint 3, and Sprint 4
- Define velocity tracking, burndown guidance, team agreements, and impediment handling
- Explain how ceremonies support sprint execution

Effort Calculation:
1. Calculate Total Work = total number of ceremonies planned across 4 sprints
2. Use Productivity = 3 ceremonies per day
3. Compute Effort: Effort (days) = Total Work / Productivity

IMPORTANT:
- You MUST show the actual numeric calculation
- You MUST ensure the numbers are mathematically correct
- Output this section clearly as:
  - Total Work = ___ ceremonies
  - Productivity = 3 ceremonies/day
  - Effort = ___ days
- At the end of your response, include a separate section titled "Effort Estimate"
CRITICAL: You MUST complete your Effort Estimate section fully before ending your response. Never cut off mid-sentence or mid-table.

Output: structured markdown with clear sections and calculations.""",

"Business_Analyst": """IMPORTANT: You are ONLY the Business Analyst.
IMPORTANT: Be concise and structured. Limit your response to essential content only. Do not repeat information. Use tables where possible. Aim for 2000-3000 words maximum.
ONLY provide YOUR role's specific output: Detailed User Stories with Given-When-Then acceptance criteria and Requirement Traceability Matrix.
Start your response with: '## BUSINESS ANALYST OUTPUT'

You are a Business Analyst in a Scrum-based campus facility project.
Responsibilities:
- Convert epics into detailed user stories
- Define acceptance criteria using Given-When-Then format
- Create requirement traceability matrix
- Identify dependencies, assumptions, constraints, and compliance concerns
- Define business rules and success criteria

Effort Calculation:
1. Calculate Total Work = total number of requirements or user stories documented
2. Use Productivity = 5 requirements per day
3. Compute Effort: Effort (days) = Total Work / Productivity

IMPORTANT:
- You MUST show the actual numeric calculation
- You MUST ensure the numbers are mathematically correct
- Output this section clearly as:
  - Total Work = ___ requirements
  - Productivity = 5 requirements/day
  - Effort = ___ days
- At the end of your response, include a separate section titled "Effort Estimate"
CRITICAL: You MUST complete your Effort Estimate section fully before ending your response. Never cut off mid-sentence or mid-table.

Output: markdown tables with calculations, traceability.""",

"UI_UX_Designer": """IMPORTANT: You are ONLY the UI/UX Designer.
IMPORTANT: Be concise and structured. Limit your response to essential content only. Do not repeat information. Use tables where possible. Aim for 2000-3000 words maximum.
ONLY provide YOUR role's specific output: User Flows, Wireframes for at least 8 screens, Design System, and WCAG 2.1 AA compliance plan.
Start your response with: '## UI/UX DESIGNER OUTPUT'

You are a UI/UX Designer in a Scrum-based campus facility project.
Responsibilities:
- Define user flows for student, instructor, and admin
- Describe wireframes or screens for key features (at least 8 screens)
- Define design system including accessibility requirements (WCAG 2.1 AA)
- Provide sprint-wise design tasks and deliverables
- Include usability considerations

Effort Calculation:
1. Calculate Total Work = total number of screens/components/tasks designed
2. Use Productivity = 3 screens or components per day
3. Compute Effort: Effort (days) = Total Work / Productivity

IMPORTANT:
- You MUST show the actual numeric calculation
- You MUST ensure the numbers are mathematically correct
- Output this section clearly as:
  - Total Work = ___ screens/components
  - Productivity = 3 screens/components/day
  - Effort = ___ days
- At the end of your response, include a separate section titled "Effort Estimate"
CRITICAL: You MUST complete your Effort Estimate section fully before ending your response. Never cut off mid-sentence or mid-table.

Output: structured markdown with calculations, tables, and detailed design breakdown.""",

"Developer": """IMPORTANT: You are ONLY the Developer.
IMPORTANT: Be concise and structured. Limit your response to essential content only. Do not repeat information. Use tables where possible. Aim for 2000-3000 words maximum.
ONLY provide YOUR role's specific output: Development Tasks with IDs, System Architecture, API Design, and Sprint-by-Sprint Implementation Plan.
Start your response with: '## DEVELOPER OUTPUT'

You are a Developer in a Scrum-based campus facility project.
Responsibilities:
- Break user stories into development tasks with IDs
- Define architecture, API tasks, data model tasks, and integration tasks
- Provide estimates, dependencies, and implementation approach
- Include frontend, backend, and AI integration work
- Define quality gates such as code review, linting, and testing

Effort Calculation:
1. Calculate Total Work = total number of story points from the Product Backlog
2. Use Velocity = 8 story points per developer per sprint
3. Number of Developers = 5
4. Team Velocity = 8 x 5 = 40 story points/sprint
5. Compute Effort: Effort (sprints) = Total Work / Team Velocity

IMPORTANT:
- You MUST use story points ONLY, NOT hours
- You MUST use ONLY the story points from the Product Backlog
- Your Total Work MUST equal the Product Backlog total exactly
- Do NOT create extra technical tasks that add more story points beyond the Product Backlog
- Map each dev task directly to a Product Backlog user story ID
- Your sprint allocation MUST follow the Product Owner's sprint plan exactly
- Sprint 1 tasks MUST only include stories planned for Sprint 1 by the Product Owner
- Sprint 2 tasks MUST only include stories planned for Sprint 2 by the Product Owner
- Sprint 3 tasks MUST only include stories planned for Sprint 3 by the Product Owner
- Sprint 4 tasks MUST only include stories planned for Sprint 4 by the Product Owner
- Each sprint's story point total MUST match the Product Owner's sprint totals exactly
- You MUST show the actual numeric calculation
- You MUST ensure the numbers are mathematically correct
- Output this section clearly as:
  - Total Work = ___ story points
  - Individual Velocity = 8 story points/developer/sprint
  - Team Velocity = 40 story points/sprint (5 developers x 8)
  - Effort = ___ sprints
- At the end of your response, include a separate section titled "Effort Estimate"
CRITICAL: You MUST complete your Effort Estimate section fully before ending your response. Never cut off mid-sentence or mid-table.

Output: markdown tables with calculations and detailed task breakdown.""",

"QA_Engineer": """IMPORTANT: You are ONLY the QA Engineer.
IMPORTANT: Be concise and structured. Limit your response to essential content only. Do not repeat information. Use tables where possible. Aim for 2000-3000 words maximum.
ONLY provide YOUR role's specific output: Test Strategy, at least 15 Functional Test Cases, at least 5 Non-Functional Test Cases, and Regression Plan.
Start your response with: '## QA ENGINEER OUTPUT'

You are a QA Engineer in a Scrum-based campus facility project.
Responsibilities:
- Define test strategy
- Create test cases with IDs (at least 15 functional + 5 non-functional)
- Include functional, non-functional, performance, security, and AI quality checks
- Define regression strategy and release quality criteria
- Align testing with sprint delivery

Effort Calculation:
1. Calculate Total Work = total number of test cases created
2. Use Productivity = 3 test cases per day
3. Compute Effort: Effort (days) = Total Work / Productivity

IMPORTANT:
- You MUST show the actual numeric calculation
- You MUST ensure the numbers are mathematically correct
- Output this section clearly as:
  - Total Work = ___ test cases
  - Productivity = 3 test cases/day
  - Effort = ___ days
- At the end of your response, include a separate section titled "Effort Estimate"
CRITICAL: You MUST complete your Effort Estimate section fully before ending your response. Never cut off mid-sentence or mid-table.

Output: markdown tables with calculations and traceability.""",

"DevOps_Engineer": """IMPORTANT: You are ONLY the DevOps Engineer.
IMPORTANT: Be concise and structured. Limit your response to essential content only. Do not repeat information. Use tables where possible. Aim for 2000-3000 words maximum.
ONLY provide YOUR role's specific output: CI/CD Pipeline Design, Infrastructure Plan, Environment Configuration, and Sprint-Aligned DevOps Tasks.
Start your response with: '## DEVOPS ENGINEER OUTPUT'

You are a DevOps Engineer in a Scrum-based campus facility project.
Responsibilities:
- Define CI/CD pipeline (GitHub Actions, Docker, Kubernetes)
- Define deployment environments and release flow (Dev, Staging, Production)
- Include monitoring, logging, rollback, and security setup
- Explain environment configuration and operational readiness
- Align DevOps work with sprint delivery

Effort Calculation:
1. Calculate Total Work = total number of DevOps tasks
2. Use Productivity = 8 tasks per day
3. Compute Effort: Effort (days) = Total Work / Productivity

IMPORTANT:
- You MUST show the actual numeric calculation
- You MUST ensure the numbers are mathematically correct
- Output this section clearly as:
  - Total Work = ___ tasks
  - Productivity = 8 tasks/day
  - Effort = ___ days
- At the end of your response, include a separate section titled "Effort Estimate"
CRITICAL: You MUST complete your Effort Estimate section fully before ending your response. Never cut off mid-sentence or mid-table.

Output: structured markdown with calculations, pipeline details, and sprint alignment."""
}

print("Scrum AI Agent prompts defined.")
for name in AGENT_SYSTEM_PROMPTS:
    print(f"  - {name}")

Scrum AI Agent prompts defined.
  - Product_Owner
  - Scrum_Master
  - Business_Analyst
  - UI_UX_Designer
  - Developer
  - QA_Engineer
  - DevOps_Engineer


#### Customer Problem Statement

In [8]:
customer_message = (
    "We need a Scrum-based project plan for our Cloud-Based Intelligent "
    "Resource Allocation System for Campus Facilities.\n\n"
    "CRITICAL INSTRUCTION: Each agent must respond ONLY with their own "
    "role-specific Scrum artifacts. Do NOT repeat or copy what the previous "
    "agent said. Each agent must produce completely unique content for their role.\n\n"
    "Product Owner: Create the Product Backlog with user stories and story points.\n"
    "Scrum Master: Define ceremonies and sprint goals for all 4 sprints.\n"
    "Business Analyst: Write detailed Given-When-Then user stories and traceability matrix.\n"
    "UI/UX Designer: Define wireframes for 8+ screens and design system.\n"
    "Developer: Break stories into dev tasks, define architecture and API design. "
    "Your total story points MUST match the Product Owner's total exactly. "
    "Do NOT add extra tasks beyond the Product Backlog scope. "
    "Map each task directly to a Product Backlog user story ID. "
    "Your sprint allocation MUST follow the Product Owner's sprint plan exactly. "
    "Each sprint's story point total MUST match the Product Owner's sprint totals. "
    "When listing sprint story point totals, you MUST add each task's story points "
    "individually and verify the sum before writing it in the table.\n"
    "QA Engineer: Create 15+ functional and 5+ non-functional test cases.\n"
    "DevOps Engineer: Design CI/CD pipeline and infrastructure plan.\n\n"
    "Project Scope: Students and faculty book rooms, labs, and equipment online. "
    "Administrators view real-time occupancy dashboards. AI/ML predicts demand and "
    "auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook, "
    "and SSO. Sends booking and conflict notifications. Generates utilization reports. "
    "Supports 5,000+ concurrent users and maintains 99.9% uptime."
)
print("Customer statement loaded.")

Customer statement loaded.


#### Run Experiment

In [9]:
agent_names = [
    "Product_Owner", "Scrum_Master", "Business_Analyst",
    "UI_UX_Designer", "Developer", "QA_Engineer", "DevOps_Engineer"
]

conversation_history = customer_message
all_results = []

print("Starting LangChain Round Robin Sequential Chain experiment...")
print("=" * 60)

start_time = time.time()

for agent_name in agent_names:
    print(f"\n>>> Running: {agent_name}")
    agent_start = time.time()

    system_prompt = AGENT_SYSTEM_PROMPTS[agent_name]

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=conversation_history)
    ]

    response = llm.invoke(messages)
    agent_response = response.content

    agent_elapsed = time.time() - agent_start

    all_results.append({
        "agent": agent_name,
        "response": agent_response,
        "time": agent_elapsed,
        "model": MODEL,
    })

    conversation_history += f"\n\n---\n\n{agent_name} Output:\n{agent_response}"

    print(f"    Done in {agent_elapsed:.1f}s | {len(agent_response):,} chars")
    print(f"    Preview: {agent_response[:100].strip()}...")

total_elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(f"  LangChain Sequential Chain completed in {total_elapsed:.1f}s")
print("=" * 60)

Starting LangChain Round Robin Sequential Chain experiment...

>>> Running: Product_Owner
    Done in 69.1s | 16,407 chars
    Preview: ## PRODUCT OWNER OUTPUT

---

# Cloud-Based Intelligent Resource Allocation System for Campus Facili...

>>> Running: Scrum_Master
    Done in 45.8s | 10,449 chars
    Preview: ## SCRUM MASTER OUTPUT

---

# Scrum Master Artifacts — Cloud-Based Intelligent Resource Allocation...

>>> Running: Business_Analyst
    Done in 105.1s | 22,348 chars
    Preview: ## BUSINESS ANALYST OUTPUT

---

# Cloud-Based Intelligent Resource Allocation System — BA Artifacts...

>>> Running: UI_UX_Designer
    Done in 100.0s | 18,165 chars
    Preview: ## UI/UX DESIGNER OUTPUT

---

# Cloud-Based Intelligent Resource Allocation System — UI/UX Design A...

>>> Running: Developer
    Done in 71.0s | 15,624 chars
    Preview: ## DEVELOPER OUTPUT

---

# Cloud-Based Intelligent Resource Allocation System — Developer Artifacts...

>>> Running: QA_Engineer
    Done in 92.7s | 16

#### Save Results

In [10]:
output_dir = "Experiment_Results_2"
os.makedirs(output_dir, exist_ok=True)

total_chars = sum(len(r["response"]) for r in all_results)

section_names = {
    "Product_Owner":    "SECTION 1 – PRODUCT OWNER (PRODUCT BACKLOG)",
    "Scrum_Master":     "SECTION 2 – SCRUM MASTER (CEREMONIES & SPRINT GOALS)",
    "Business_Analyst": "SECTION 3 – BUSINESS ANALYST (USER STORIES & TRACEABILITY)",
    "UI_UX_Designer":   "SECTION 4 – UI/UX DESIGNER (WIREFRAMES & DESIGN SYSTEM)",
    "Developer":        "SECTION 5 – DEVELOPER (SPRINT TASKS & ARCHITECTURE)",
    "QA_Engineer":      "SECTION 6 – QA ENGINEER (TEST PLAN)",
    "DevOps_Engineer":  "SECTION 7 – DEVOPS ENGINEER (CI/CD & INFRASTRUCTURE)",
}

for result in all_results:
    filepath = os.path.join(output_dir, f"{result['agent']}_output.md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {result['agent'].replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {result['model']} | **Time:** {result['time']:.1f}s\n\n")
        f.write("---\n\n")
        f.write(result["response"])
    print(f"Saved: {filepath}")

combined_path = os.path.join(output_dir, "Phase2_Scrum_Complete_Report.md")
with open(combined_path, "w", encoding="utf-8") as f:
    f.write("# Phase 2 – Scrum Project Plan\n")
    f.write("# Cloud-Based Intelligent Resource Allocation System for Campus Facilities\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**LLM Model:** {MODEL}\n")
    f.write("**Framework:** LangChain – Round Robin Sequential Chain\n")
    f.write("**Methodology:** Scrum (Agile Iterative Development)\n\n---\n\n")
    f.write("## SECTION 0 – CUSTOMER PROBLEM STATEMENT\n\n")
    f.write(customer_message + "\n\n---\n\n")
    for result in all_results:
        section = section_names.get(result["agent"], result["agent"])
        f.write(f"## {section}\n\n")
        f.write(f"**Model:** {result['model']} | **Time:** {result['time']:.1f}s\n\n")
        f.write(result["response"] + "\n\n---\n\n")

metadata = {
    "experiment": {
        "title": "Phase 2 – Scrum Project Planning",
        "project": "Cloud-Based Intelligent Resource Allocation System",
        "model": MODEL,
        "framework": "LangChain – Round Robin Sequential Chain",
        "date": datetime.now().isoformat(),
        "total_time_seconds": round(total_elapsed, 2),
        "total_chars": total_chars,
    },
    "agents": [
        {"name": r["agent"], "time": round(r["time"], 2), "model": r["model"]}
        for r in all_results
    ],
}
metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"\nCombined report saved: {combined_path}")
print(f"Metadata saved: {metadata_path}")
print(f"Total characters used: {total_chars:,}")

Saved: Experiment_Results_2\Product_Owner_output.md
Saved: Experiment_Results_2\Scrum_Master_output.md
Saved: Experiment_Results_2\Business_Analyst_output.md
Saved: Experiment_Results_2\UI_UX_Designer_output.md
Saved: Experiment_Results_2\Developer_output.md
Saved: Experiment_Results_2\QA_Engineer_output.md
Saved: Experiment_Results_2\DevOps_Engineer_output.md

Combined report saved: Experiment_Results_2\Phase2_Scrum_Complete_Report.md
Metadata saved: Experiment_Results_2\experiment_metadata.json
Total characters used: 118,008


#### Work Effort, Duration & Effort Analysis

In [11]:
HOURS_PER_PT = 4
WD = 8
dur_min = total_elapsed / 60
avg_per_agent = total_elapsed / len(all_results) if all_results else 0
NUM_SPRINTS = 4
SPRINT_WEEKS = 2
proj_weeks = NUM_SPRINTS * SPRINT_WEEKS
proj_budget_hrs = proj_weeks * 5 * WD

print('=' * 62)
print('  WORK EFFORT, DURATION & EFFORT ANALYSIS')
print('=' * 62)
print(f'\n📅 DURATION')
print(f'  Sprints Planned         : {NUM_SPRINTS} x {SPRINT_WEEKS}-week sprints')
print(f'  Total Project Duration  : {proj_weeks} weeks ({proj_weeks*5} working days / {proj_weeks*7} calendar days)')
print(f'  Capacity Budget         : {proj_budget_hrs} work-hours')
print(f'  AI Experiment (wall-clock): {total_elapsed:.1f}s  ({dur_min:.1f} min)')
print(f'  Avg Time per Agent      : {avg_per_agent:.1f}s')

po = next((r['response'] for r in all_results if r['agent'] == 'Product_Owner'), '')

# Try regex patterns
total_sp_match = re.search(r'Total Work\s*[=:|]\s*\*?\*?(\d+)', po, re.IGNORECASE)
if not total_sp_match:
    total_sp_match = re.search(r'\*\*(\d+)\*\*\s*story points', po, re.IGNORECASE)
if not total_sp_match:
    total_sp_match = re.search(r'(\d+)\s*story points', po, re.IGNORECASE)

# If still not found, sum sprint totals from sprint table
if not total_sp_match:
    sprint_totals = re.findall(r'\|\s*\*\*Total\*\*\s*\|[^|]*\|\s*\*?\*?(\d+)', po, re.IGNORECASE)
    if not sprint_totals:
        sprint_totals = re.findall(r'\|\s*\*\*Sprint \d+\*\*[^|]*\|\s*\*?\*?(\d+)\s*\*?\*?\s*\|', po, re.IGNORECASE)
    if not sprint_totals:
        sprint_totals = re.findall(r'\|\s*\*\*(\d+)\*\*\s*\|', po)
    total_sp = sum(int(x) for x in sprint_totals) if sprint_totals else 0
else:
    total_sp = int(total_sp_match.group(1))

avg_vel = total_sp / NUM_SPRINTS if total_sp else 0

print(f'\n🎯 WORK EFFORT  (Story Points)')
print(f'  Total Story Points      : {total_sp} pts')
print(f'  Team Velocity           : 40 story points/sprint')
print(f'  Sprints Needed          : {NUM_SPRINTS} sprints')
print(f'  Avg Velocity            : {avg_vel:.1f} pts/sprint')

print(f'\n⏱  EFFORT ESTIMATION  (@ {HOURS_PER_PT}h per story point)')
th = total_sp * HOURS_PER_PT
print(f'  Total Estimated Effort  : {th} hrs  ({th/WD:.1f} days)  ({th/WD/5:.1f} weeks)')
print(f'  Capacity Utilisation    : {th/proj_budget_hrs*100:.1f}%   ({th}/{proj_budget_hrs} hrs)')

print(f'\n🤖 AGENT TIME EFFORT')
print(f"  {'Agent':<22} {'Time(s)':>8} {'%':>6} {'Chars':>8}")
print(f"  {'-'*50}")
for r in all_results:
    pct = r['time'] / total_elapsed * 100 if total_elapsed else 0
    print(f"  {r['agent']:<22} {r['time']:>8.1f}  {pct:>5.1f}%  {len(r['response']):>8,}")
print(f"  {'-'*50}")
print(f"  {'TOTAL':<22} {total_elapsed:>8.1f}  100.0%  {total_chars:>8,}")
print(f"  Throughput : N/A (LangChain does not expose token counts directly)")
print(f"  Estimated Cost: ~$0.005 USD (Claude Sonnet 4.6)")
print('=' * 62)

  WORK EFFORT, DURATION & EFFORT ANALYSIS

📅 DURATION
  Sprints Planned         : 4 x 2-week sprints
  Total Project Duration  : 8 weeks (40 working days / 56 calendar days)
  Capacity Budget         : 320 work-hours
  AI Experiment (wall-clock): 564.5s  (9.4 min)
  Avg Time per Agent      : 80.6s

🎯 WORK EFFORT  (Story Points)
  Total Story Points      : 141 pts
  Team Velocity           : 40 story points/sprint
  Sprints Needed          : 4 sprints
  Avg Velocity            : 35.2 pts/sprint

⏱  EFFORT ESTIMATION  (@ 4h per story point)
  Total Estimated Effort  : 564 hrs  (70.5 days)  (14.1 weeks)
  Capacity Utilisation    : 176.2%   (564/320 hrs)

🤖 AGENT TIME EFFORT
  Agent                   Time(s)      %    Chars
  --------------------------------------------------
  Product_Owner              69.1   12.2%    16,407
  Scrum_Master               45.8    8.1%    10,449
  Business_Analyst          105.1   18.6%    22,348
  UI_UX_Designer            100.0   17.7%    18,165
  Develop

#### Experiment Summary

In [12]:
print('=' * 62)
print('  SCRUM EXPERIMENT COMPLETE')
print('=' * 62)
print()
print(f'  Model       : {MODEL}')
print(f'  Framework   : LangChain')
print(f'  Method      : Round Robin Sequential Chain')
print(f'  Total Agents: {len(all_results)}')
print(f'  Total Time   : {total_elapsed:.1f}s  ({total_elapsed/60:.1f} min)')
print(f'  Avg/Agent   : {total_elapsed/len(all_results):.1f}s')
print()

po_response = next((r['response'] for r in all_results if r['agent'] == 'Product_Owner'), '')

# Try regex patterns
total_sp_match = re.search(r'Total Work\s*[=:|]\s*\*?\*?(\d+)', po_response, re.IGNORECASE)
if not total_sp_match:
    total_sp_match = re.search(r'\*\*(\d+)\*\*\s*story points', po_response, re.IGNORECASE)
if not total_sp_match:
    total_sp_match = re.search(r'(\d+)\s*story points', po_response, re.IGNORECASE)

# If still not found, sum sprint totals from sprint table
if not total_sp_match:
    sprint_totals = re.findall(r'\|\s*\*\*Total\*\*\s*\|[^|]*\|\s*\*?\*?(\d+)', po_response, re.IGNORECASE)
    if not sprint_totals:
        sprint_totals = re.findall(r'\|\s*\*\*Sprint \d+\*\*[^|]*\|\s*\*?\*?(\d+)\s*\*?\*?\s*\|', po_response, re.IGNORECASE)
    if not sprint_totals:
        sprint_totals = re.findall(r'\|\s*\*\*(\d+)\*\*\s*\|', po_response)
    total_sp = sum(int(x) for x in sprint_totals) if sprint_totals else 0
else:
    total_sp = int(total_sp_match.group(1))

num_sprints = 4
sprint_weeks = 2
total_weeks = num_sprints * sprint_weeks
total_working_days = total_weeks * 5
total_calendar_days = total_weeks * 7

print(f'  -- Work Effort --')
print(f'  Total Story Points : {total_sp} pts')
print(f'  Team Velocity      : 40 story points/sprint')
print(f'  Sprints Needed     : {num_sprints} sprints')
print(f'  Total Effort       : {total_sp * 4} hrs  ({total_sp * 4 / 8:.1f} days)')
print(f'  Avg Velocity       : {total_sp / num_sprints:.1f} pts/sprint')
print()
print(f'  -- Duration --')
print(f'  Scrum Plan  : {num_sprints} sprints x {sprint_weeks} weeks = {total_weeks} weeks')
print(f'  Working Days: {total_working_days} working days / {total_calendar_days} calendar days')
print(f'  AI Experiment: {total_elapsed:.1f}s  ({total_elapsed/60:.1f} min)')
print(f'  Avg/Agent   : {total_elapsed/len(all_results):.1f}s')
print()
print(f'  -- Agent Breakdown --')
for r in all_results:
    print(f"    {r['agent']:<22s} | Time: {r['time']:>6.1f}s | Chars: {len(r['response']):>6,}")
print()
print(f'  Total Characters Used: {total_chars:,}')
print(f'  Output Dir: Experiment_Results_2/')
print('=' * 62)

  SCRUM EXPERIMENT COMPLETE

  Model       : claude-sonnet-4-6
  Framework   : LangChain
  Method      : Round Robin Sequential Chain
  Total Agents: 7
  Total Time   : 564.5s  (9.4 min)
  Avg/Agent   : 80.6s

  -- Work Effort --
  Total Story Points : 141 pts
  Team Velocity      : 40 story points/sprint
  Sprints Needed     : 4 sprints
  Total Effort       : 564 hrs  (70.5 days)
  Avg Velocity       : 35.2 pts/sprint

  -- Duration --
  Scrum Plan  : 4 sprints x 2 weeks = 8 weeks
  Working Days: 40 working days / 56 calendar days
  AI Experiment: 564.5s  (9.4 min)
  Avg/Agent   : 80.6s

  -- Agent Breakdown --
    Product_Owner          | Time:   69.1s | Chars: 16,407
    Scrum_Master           | Time:   45.8s | Chars: 10,449
    Business_Analyst       | Time:  105.1s | Chars: 22,348
    UI_UX_Designer         | Time:  100.0s | Chars: 18,165
    Developer              | Time:   71.0s | Chars: 15,624
    QA_Engineer            | Time:   92.7s | Chars: 16,947
    DevOps_Engineer       